In [2]:
# 导入所需的库
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
import nltk
from nltk.corpus import stopwords
import re

# 加载数据集的函数，输入是json文件路径，输出是DataFrame格式的数据
def load_data(json_file):
    # 打开json文件并加载数据
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 初始化用于存储数据的列表
    ids = []
    claim_texts = []
    claim_labels = []

    # 遍历json数据，提取id、claim_text和claim_label，并添加到相应列表中
    for claim_id, claim_data in data.items():
        ids.append(claim_id.split('-')[1])
        claim_texts.append(claim_data["claim_text"])
        claim_labels.append(claim_data["claim_label"])

    # 创建DataFrame对象，列名为'id'、'claim_text'和'claim_label'，并返回该对象
    df = pd.DataFrame({
        'id': ids,
        'claim_text': claim_texts,
        'claim_label': claim_labels
    })

    return df

# 加载训练数据，调用load_data函数，将数据存储在train_df中
train_df = load_data('../data/train-claims.json')

# 标签编码，将标签（claim_label）转换为数字编码，并将编码后的结果存储在'claim_encoded'列中
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['claim_label'])

# 划分训练集和验证集，使用train_test_split函数将数据集划分为训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(train_df['claim_text'], train_df['label_encoded'], test_size=0.2, random_state=42)

# 文本预处理：转换为小写和去除停用词
stop_words = set(stopwords.words('english'))  # 加载英文停用词表
X_train = X_train.apply(lambda x: ' '.join([word.lower() for word in x.split() if word.lower() not in stop_words]))
X_val = X_val.apply(lambda x: ' '.join([word.lower() for word in x.split() if word.lower() not in stop_words]))

# 文本向量化，使用Tokenizer对文本进行向量化处理
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

# 将训练集和验证集的文本数据转换为序列
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)

# 填充序列，使所有序列的长度相同，使用pad_sequences函数对序列进行填充
max_length = max([len(seq) for seq in X_train_seq])
X_train_padded = pad_sequences(X_train_seq, maxlen=max_length, padding='post')
X_val_padded = pad_sequences(X_val_seq, maxlen=max_length, padding='post')

# 构建RNN模型，使用Sequential模型搭建RNN模型的结构
model = Sequential([
    Embedding(input_dim=len(tokenizer.word_index)+1, output_dim=100, input_length=max_length),
    LSTM(128),
    Dense(64, activation='relu'),
    Dense(len(label_encoder.classes_), activation='softmax')
])

# 编译模型，配置模型的学习过程
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 定义早停策略，当验证集上的损失不再下降时提前停止训练
early_stopping = EarlyStopping(patience=3, restore_best_weights=True)

# 训练模型，使用fit方法对模型进行训练
history = model.fit(X_train_padded, y_train, epochs=20, batch_size=32, validation_data=(X_val_padded, y_val), callbacks=[early_stopping])

# 评估模型，在验证集上评估模型的性能表现
val_loss, val_accuracy = model.evaluate(X_val_padded, y_val)
print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_accuracy)


Epoch 1/20
31/31 [==============================] - 2s 32ms/step - loss: 1.2818 - accuracy: 0.4114 - val_loss: 1.2574 - val_accuracy: 0.4512
Epoch 2/20
31/31 [==============================] - 1s 21ms/step - loss: 1.2656 - accuracy: 0.4155 - val_loss: 1.2531 - val_accuracy: 0.4512
Epoch 3/20
31/31 [==============================] - 1s 23ms/step - loss: 1.2584 - accuracy: 0.4155 - val_loss: 1.2568 - val_accuracy: 0.4512
Epoch 4/20
31/31 [==============================] - 1s 21ms/step - loss: 1.2583 - accuracy: 0.4155 - val_loss: 1.2546 - val_accuracy: 0.4512
Epoch 5/20
8/8 [==============================] - 0s 5ms/step - loss: 1.2531 - accuracy: 0.4512
Validation Loss: 1.2531452178955078
Validation Accuracy: 0.45121949911117554


In [2]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# 加载数据集的函数，输入是json文件路径，输出是DataFrame格式的数据
def load_data(json_file):
    # 打开json文件并加载数据
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # 初始化用于存储数据的列表
    ids = []
    claim_texts = []
    claim_labels = []

    # 遍历json数据，提取id、claim_text和claim_label，并添加到相应列表中
    for claim_id, claim_data in data.items():
        ids.append(claim_id.split('-')[1])
        claim_texts.append(claim_data["claim_text"])
        claim_labels.append(claim_data["claim_label"])

    # 创建DataFrame对象，列名为'id'、'claim_text'和'claim_label'，并返回该对象
    df = pd.DataFrame({
        'id': ids,
        'claim_text': claim_texts,
        'claim_label': claim_labels
    })

    return df

# 加载训练数据，调用load_data函数，将数据存储在train_df中
train_df = load_data('data/train-claims.json')

# 标签编码，将标签（claim_label）转换为数字编码，并将编码后的结果存储在'claim_encoded'列中
label_encoder = LabelEncoder()
train_df['label_encoded'] = label_encoder.fit_transform(train_df['claim_label'])

# 清除特殊字符和标点符号
train_df['claim_text'] = train_df['claim_text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

# 分词，并转换为小写
train_df['claim_text'] = train_df['claim_text'].apply(lambda x: word_tokenize(x.lower()))

# 去除停用词
stop_words = set(stopwords.words('english'))
train_df['claim_text'] = train_df['claim_text'].apply(lambda x: [word for word in x if word not in stop_words])

# 文本向量化
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_df['claim_text'])
X_seq = tokenizer.texts_to_sequences(train_df['claim_text'])

# 填充序列
max_length = max([len(seq) for seq in X_seq])
X_padded = pad_sequences(X_seq, maxlen=max_length, padding='post')

# 构建RNN模型
def build_rnn_model(input_dim, output_dim, max_length, num_classes):
    model = Sequential([
        Embedding(input_dim=input_dim, output_dim=output_dim, input_length=max_length),
        LSTM(128),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    return model

# 定义早停策略
early_stopping = EarlyStopping(patience=3, restore_best_weights=True)

# 训练并评估模型
def train_and_evaluate_model(X, y, input_dim, output_dim, max_length, num_classes):
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = build_rnn_model(input_dim, output_dim, max_length, num_classes)
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    
    history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_val, y_val), callbacks=[early_stopping])
    
    val_loss, val_accuracy = model.evaluate(X_val, y_val)
    print("Validation Loss:", val_loss)
    print("Validation Accuracy:", val_accuracy)
    
    return model

# 训练并评估模型
trained_model = train_and_evaluate_model(X_padded, train_df['label_encoded'], len(tokenizer.word_index) + 1, 100, max_length, len(label_encoder.classes_))


Epoch 1/20
31/31 [==============================] - 3s 40ms/step - loss: 1.2973 - accuracy: 0.3778 - val_loss: 1.2537 - val_accuracy: 0.4512
Epoch 2/20
31/31 [==============================] - 1s 26ms/step - loss: 1.2619 - accuracy: 0.3971 - val_loss: 1.2539 - val_accuracy: 0.4512
Epoch 3/20
31/31 [==============================] - 1s 24ms/step - loss: 1.2671 - accuracy: 0.4155 - val_loss: 1.2513 - val_accuracy: 0.4512
Epoch 4/20
31/31 [==============================] - 1s 30ms/step - loss: 1.2565 - accuracy: 0.4155 - val_loss: 1.2534 - val_accuracy: 0.4512
Epoch 5/20
31/31 [==============================] - 1s 28ms/step - loss: 1.1914 - accuracy: 0.4308 - val_loss: 1.2586 - val_accuracy: 0.4512
Epoch 6/20
8/8 [==============================] - 0s 8ms/step - loss: 1.2513 - accuracy: 0.4512
Validation Loss: 1.251299262046814
Validation Accuracy: 0.45121949911117554
